# Day 60: Autonomous Perception I — Object Detection
## 100 ML Projects | Deep Learning Capstone (Part 1 of 3)

**Project:** Autonomous Perception Ensemble

**Framework:** PyTorch + Ultralytics

**Model:** YOLOv8-nano

**Dataset:** BDD100K (subset)

---

### Capstone Overview

```
┌─────────────────────────────────────────────────────────────┐
│                   AUTONOMOUS PERCEPTION                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Day 60: Detection ◄── You are here                       │
│   Day 61: Segmentation                                     │
│   Day 62: Depth + Fusion + ONNX                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

In [ ]:
# Install dependencies
!pip install ultralytics -q
!pip install roboflow -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import time
from PIL import Image

import torch
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Create save directory
SAVE_PATH = '/content/drive/MyDrive/100-days-of-ml/capstone-autonomous-perception'
os.makedirs(SAVE_PATH, exist_ok=True)

print(f"Save path: {SAVE_PATH}")

---
## 1. Autonomous Perception Overview

In [ ]:
print("""
AUTONOMOUS PERCEPTION PIPELINE
===============================

Self-driving cars need multiple perception systems working together:

┌─────────────────────────────────────────────────────────────┐
│                      CAMERA INPUT                           │
└─────────────────────────┬───────────────────────────────────┘
                          │
        ┌─────────────────┼─────────────────┐
        ▼                 ▼                 ▼
┌───────────────┐ ┌───────────────┐ ┌───────────────┐
│   DETECTION   │ │ SEGMENTATION  │ │    DEPTH      │
│               │ │               │ │               │
│ "What's here" │ │ "Where's the  │ │ "How far is   │
│               │ │    road?"     │ │   everything" │
│  Cars: 3      │ │  ████ road    │ │  Car: 12m     │
│  People: 1    │ │  ░░░░ sidewlk │ │  Sign: 8m     │
│  Signs: 2     │ │               │ │               │
└───────────────┘ └───────────────┘ └───────────────┘
        │                 │                 │
        └─────────────────┼─────────────────┘
                          ▼
              ┌───────────────────┐
              │      FUSION       │
              │                   │
              │ Combined scene    │
              │ understanding     │
              └───────────────────┘

TODAY: Object Detection with YOLOv8
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Detect: cars, trucks, pedestrians, cyclists, traffic signs/lights
""")

---
## 2. BDD100K Dataset

In [ ]:
print("""
BDD100K DATASET
================

Berkeley DeepDrive 100K — largest driving dataset

Full dataset:
  • 100,000 video clips
  • 10 tasks (detection, segmentation, tracking, etc.)
  • Diverse: weather, time of day, locations
  
We're using: Roboflow BDD100K subset (~10K images)
  • Pre-formatted for YOLO
  • ~2GB download
  • Detection labels ready

Classes for detection:
  • car, truck, bus
  • person, rider (cyclist/motorcyclist)
  • bicycle, motorcycle
  • traffic light, traffic sign
  • train
""")

---
## 3. Download Dataset

In [ ]:
# Check if dataset already exists in Drive
DATASET_PATH = f"{SAVE_PATH}/bdd100k-detection"

if os.path.exists(f"{DATASET_PATH}/data.yaml"):
    print("Dataset already downloaded! Loading from Drive...")
else:
    print("Downloading BDD100K detection subset from Roboflow...")
    
    from roboflow import Roboflow
    
    # Using public BDD100K subset
    # Note: You may need to create a free Roboflow account
    rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Get free key at roboflow.com
    project = rf.workspace("bdd-100k").project("bdd100k-nit3m")
    dataset = project.version(1).download("yolov8", location=DATASET_PATH)
    
    print(f"Downloaded to: {DATASET_PATH}")

In [ ]:
# Alternative: Direct download from a public source
# If Roboflow doesn't work, use this backup approach

import urllib.request
import zipfile

# Check if we need to download
if not os.path.exists(f"{DATASET_PATH}/data.yaml"):
    print("Using alternative download method...")
    
    # Create directory
    os.makedirs(DATASET_PATH, exist_ok=True)
    
    # For demonstration, we'll use a smaller BDD subset
    # In practice, you'd download from Roboflow or BDD official site
    print("""
    MANUAL DOWNLOAD OPTION:
    
    1. Go to: https://universe.roboflow.com/bdd-100k/bdd100k-nit3m
    2. Click 'Download' → Select 'YOLOv8' format
    3. Upload the zip to your Google Drive
    4. Extract to: {}
    """.format(DATASET_PATH))

In [ ]:
# Explore dataset structure
print("Dataset structure:")
!ls -la {DATASET_PATH}

In [ ]:
# Check data.yaml
import yaml

with open(f"{DATASET_PATH}/data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset config:")
print(f"  Train: {data_config.get('train', 'N/A')}")
print(f"  Val: {data_config.get('val', 'N/A')}")
print(f"  Classes: {data_config.get('names', 'N/A')}")
print(f"  Num classes: {data_config.get('nc', 'N/A')}")

In [ ]:
# Count images
train_images = glob.glob(f"{DATASET_PATH}/train/images/*.jpg")
val_images = glob.glob(f"{DATASET_PATH}/valid/images/*.jpg")

print(f"Training images: {len(train_images)}")
print(f"Validation images: {len(val_images)}")

---
## 4. Explore Dataset

In [ ]:
def visualize_sample(image_path, label_path, class_names):
    """Visualize image with YOLO format bounding boxes"""
    # Load image
    img = Image.open(image_path)
    img_w, img_h = img.size
    
    # Load labels (YOLO format: class x_center y_center width height)
    boxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    x_center, y_center, w, h = map(float, parts[1:5])
                    
                    # Convert to pixel coordinates
                    x1 = (x_center - w/2) * img_w
                    y1 = (y_center - h/2) * img_h
                    x2 = (x_center + w/2) * img_w
                    y2 = (y_center + h/2) * img_h
                    
                    boxes.append((cls_id, x1, y1, x2, y2))
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img)
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))
    
    for cls_id, x1, y1, x2, y2 in boxes:
        color = colors[cls_id % len(colors)]
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                              fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)
        
        label = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
        ax.text(x1, y1-5, label, color='white', fontsize=10,
                bbox=dict(boxstyle='round', facecolor=color, alpha=0.8))
    
    ax.set_title(f"Sample: {os.path.basename(image_path)} ({len(boxes)} objects)")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Visualization function defined")

In [ ]:
# Visualize random samples
class_names = data_config.get('names', [])
print(f"Classes: {class_names}\n")

sample_indices = np.random.choice(len(train_images), 4, replace=False)

for idx in sample_indices:
    img_path = train_images[idx]
    label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    visualize_sample(img_path, label_path, class_names)

In [ ]:
# Class distribution
from collections import Counter

class_counts = Counter()

for img_path in train_images:
    label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counts[int(parts[0])] += 1

# Plot distribution
fig, ax = plt.subplots(figsize=(12, 5))

classes = [class_names[i] if i < len(class_names) else f"class_{i}" 
           for i in sorted(class_counts.keys())]
counts = [class_counts[i] for i in sorted(class_counts.keys())]

bars = ax.bar(classes, counts, color='steelblue', edgecolor='black')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_title('BDD100K Detection - Class Distribution')
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nTotal objects: {sum(counts):,}")

---
## 5. YOLOv8 Architecture

In [ ]:
print("""
YOLOv8 ARCHITECTURE
====================

YOLO = "You Only Look Once"
Single forward pass → all detections

┌─────────────────────────────────────────────────────────────┐
│                         YOLOv8                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   INPUT         BACKBONE           NECK            HEAD     │
│                                                             │
│   ┌───────┐    ┌─────────┐     ┌─────────┐    ┌─────────┐  │
│   │       │    │ CSPDark │     │   PANet │    │ Detect  │  │
│   │ Image │ ──►│   net   │ ──► │   FPN   │ ──►│  Head   │  │
│   │640×640│    │         │     │         │    │         │  │
│   └───────┘    └─────────┘     └─────────┘    └─────────┘  │
│                                                     │       │
│                                                     ▼       │
│                                              ┌──────────┐   │
│                                              │ Boxes +  │   │
│                                              │ Classes  │   │
│                                              └──────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘

YOLOv8 Variants:
┌──────────┬────────────┬─────────────┬───────────┐
│ Model    │ Parameters │ Speed (ms)  │ mAP       │
├──────────┼────────────┼─────────────┼───────────┤
│ YOLOv8n  │ 3.2M       │ 1.2         │ 37.3      │  ◄── We use this
│ YOLOv8s  │ 11.2M      │ 1.9         │ 44.9      │
│ YOLOv8m  │ 25.9M      │ 3.4         │ 50.2      │
│ YOLOv8l  │ 43.7M      │ 5.3         │ 52.9      │
│ YOLOv8x  │ 68.2M      │ 8.4         │ 53.9      │
└──────────┴────────────┴─────────────┴───────────┘

Why YOLOv8-nano?
  • Fast training on Colab
  • Small ONNX export (~6MB)
  • Good enough for perception demo
  • Easy deployment
""")

---
## 6. Load Pretrained YOLOv8

In [ ]:
# Load YOLOv8-nano pretrained on COCO
model = YOLO('yolov8n.pt')

print("Model loaded!")
print(f"\nModel info:")
print(f"  Type: YOLOv8-nano")
print(f"  Pretrained: COCO (80 classes)")
print(f"  Parameters: ~3.2M")

In [ ]:
# Test inference on a sample before fine-tuning
sample_img = train_images[0]

results = model.predict(sample_img, verbose=False)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original
axes[0].imshow(Image.open(sample_img))
axes[0].set_title('Original')
axes[0].axis('off')

# With COCO detections
result_img = results[0].plot()
axes[1].imshow(result_img)
axes[1].set_title('YOLOv8n (COCO pretrained) - Before fine-tuning')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("Note: Using COCO classes, not BDD100K classes yet")

---
## 7. Fine-tune on BDD100K

In [ ]:
# Check if we already have a fine-tuned model
FINETUNED_MODEL_PATH = f"{SAVE_PATH}/yolov8n_bdd100k.pt"

if os.path.exists(FINETUNED_MODEL_PATH):
    print("Found existing fine-tuned model! Loading...")
    model = YOLO(FINETUNED_MODEL_PATH)
    SKIP_TRAINING = True
else:
    print("No existing model found. Will train from scratch.")
    SKIP_TRAINING = False

In [ ]:
# Training configuration
EPOCHS = 30
IMG_SIZE = 640
BATCH_SIZE = 16

print(f"""
Training Configuration:
  Epochs: {EPOCHS}
  Image size: {IMG_SIZE}×{IMG_SIZE}
  Batch size: {BATCH_SIZE}
  Optimizer: SGD (default)
  LR: 0.01 (default)
""")

In [ ]:
# Fine-tune!
if not SKIP_TRAINING:
    print("Starting fine-tuning...")
    print("="*60)
    
    results = model.train(
        data=f"{DATASET_PATH}/data.yaml",
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=10,  # Early stopping
        save=True,
        project=SAVE_PATH,
        name='yolov8n_bdd100k_training',
        exist_ok=True,
        pretrained=True,
        verbose=True
    )
    
    print("\nTraining complete!")
else:
    print("Skipping training - using cached model")

In [ ]:
# Load best model
if not SKIP_TRAINING:
    best_model_path = f"{SAVE_PATH}/yolov8n_bdd100k_training/weights/best.pt"
    model = YOLO(best_model_path)
    
    # Copy to main save location
    import shutil
    shutil.copy(best_model_path, FINETUNED_MODEL_PATH)
    print(f"Best model saved to: {FINETUNED_MODEL_PATH}")

print("Model ready for inference!")

---
## 8. Evaluate Model

In [ ]:
# Validate on validation set
print("Running validation...")

metrics = model.val(
    data=f"{DATASET_PATH}/data.yaml",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    verbose=False
)

print(f"""
Validation Results:
  mAP@50: {metrics.box.map50:.4f}
  mAP@50-95: {metrics.box.map:.4f}
  Precision: {metrics.box.mp:.4f}
  Recall: {metrics.box.mr:.4f}
""")

In [ ]:
# Per-class AP
print("Per-class AP@50:")
print("-" * 30)

for i, ap in enumerate(metrics.box.ap50):
    class_name = class_names[i] if i < len(class_names) else f"class_{i}"
    print(f"  {class_name:15s}: {ap:.4f}")

---
## 9. Visualize Predictions

In [ ]:
# Predict on validation images
sample_val_images = np.random.choice(val_images, 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, img_path in enumerate(sample_val_images):
    results = model.predict(img_path, verbose=False, conf=0.25)
    result_img = results[0].plot()
    
    axes[i].imshow(result_img)
    axes[i].set_title(f"{len(results[0].boxes)} detections")
    axes[i].axis('off')

plt.suptitle('YOLOv8 BDD100K Detections', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed view of one prediction
sample_img = val_images[np.random.randint(len(val_images))]
results = model.predict(sample_img, verbose=False, conf=0.25)

print(f"Image: {os.path.basename(sample_img)}")
print(f"Detections: {len(results[0].boxes)}")
print("\nDetected objects:")
print("-" * 50)

for box in results[0].boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    
    class_name = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
    print(f"  {class_name:15s} | Conf: {conf:.2f} | Box: ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")

# Show image
plt.figure(figsize=(14, 8))
plt.imshow(results[0].plot())
plt.axis('off')
plt.title('Detailed Detection View')
plt.show()

---
## 10. Inference Speed Benchmark

In [ ]:
# Benchmark inference speed
import time

n_runs = 50
test_images = np.random.choice(val_images, n_runs, replace=True)

# Warmup
for _ in range(5):
    _ = model.predict(test_images[0], verbose=False)

# Benchmark
times = []
for img_path in test_images:
    start = time.time()
    _ = model.predict(img_path, verbose=False)
    times.append(time.time() - start)

avg_time = np.mean(times) * 1000  # Convert to ms
fps = 1000 / avg_time

print(f"""
Inference Speed (PyTorch):
  Average: {avg_time:.1f} ms/image
  FPS: {fps:.1f}
  Min: {min(times)*1000:.1f} ms
  Max: {max(times)*1000:.1f} ms
""")

---
## 11. Export to ONNX (Preview)

In [ ]:
# Export to ONNX (we'll do full optimization in Day 62)
print("Exporting to ONNX...")

onnx_path = model.export(
    format='onnx',
    imgsz=IMG_SIZE,
    simplify=True,
    dynamic=False
)

print(f"\nONNX model exported!")

In [ ]:
# Check ONNX file size
import os

onnx_file = f"{FINETUNED_MODEL_PATH.replace('.pt', '.onnx')}"

# Find the exported file
if os.path.exists(onnx_path):
    onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"ONNX file: {onnx_path}")
    print(f"Size: {onnx_size:.1f} MB")
    
    # Copy to save path
    import shutil
    shutil.copy(onnx_path, f"{SAVE_PATH}/yolov8n_bdd100k.onnx")
    print(f"\nCopied to: {SAVE_PATH}/yolov8n_bdd100k.onnx")

---
## 12. Save Artifacts

In [ ]:
# Save class names for Day 61 & 62
import json

artifacts = {
    'class_names': class_names,
    'num_classes': len(class_names),
    'img_size': IMG_SIZE,
    'model_path': FINETUNED_MODEL_PATH,
    'onnx_path': f"{SAVE_PATH}/yolov8n_bdd100k.onnx",
    'dataset_path': DATASET_PATH,
    'metrics': {
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr)
    }
}

with open(f"{SAVE_PATH}/day60_artifacts.json", 'w') as f:
    json.dump(artifacts, f, indent=2)

print("Artifacts saved!")
print(f"\nSaved to: {SAVE_PATH}/day60_artifacts.json")

In [ ]:
# Summary of saved files
print("""
DAY 60 OUTPUTS
===============

Saved to Google Drive:
""")
print(f"  {SAVE_PATH}/")
print(f"    ├── yolov8n_bdd100k.pt      (fine-tuned weights)")
print(f"    ├── yolov8n_bdd100k.onnx    (ONNX export)")
print(f"    ├── day60_artifacts.json    (metadata)")
print(f"    └── bdd100k-detection/      (dataset)")
print(f"""
For Day 61:
  • Load yolov8n_bdd100k.pt for detection
  • Use same BDD100K dataset for segmentation
  • Add drivable area segmentation with U-Net
""")

---
## Summary

### Day 60: Object Detection ✓

```
┌─────────────────────────────────────────┐
│     AUTONOMOUS PERCEPTION ENSEMBLE      │
├─────────────────────────────────────────┤
│                                         │
│   [✓] Day 60: Detection (YOLOv8)        │
│   [ ] Day 61: Segmentation (U-Net)      │
│   [ ] Day 62: Depth + Fusion + ONNX     │
│                                         │
└─────────────────────────────────────────┘
```

### What We Built

| Component | Details |
|-----------|--------|
| Model | YOLOv8-nano |
| Dataset | BDD100K (~10K images) |
| Classes | car, truck, bus, person, cyclist, etc. |
| Training | 30 epochs, fine-tuned from COCO |
| Export | ONNX ready |

### Key Metrics

| Metric | Value |
|--------|-------|
| mAP@50 | ~0.5+ |
| Inference | ~20-30ms |
| Model Size | ~6MB (ONNX) |

### Next: Day 61
- Load this detection model
- Train U-Net for drivable area segmentation
- Same BDD100K dataset

---

**Detection model ready for fusion! 🚗**